# CellRank trajectory inference: RNA velocity + moscot, alone and combined

**Cell set.** The kernel-weighting study runs on the 68,166 **devbrain** cells. RNA velocity
needs spliced/unspliced counts, which exist only for these (Brooks has `norm_counts`/
`raw_counts` only), and CellRank requires every kernel in a combination to be defined over
the same cells. The graph is built on `X_pca_harmony_joint` - the Harmony embedding from the
joint devbrain+Brooks run - so root-cell selection still benefits from the Brooks bridge even
though Brooks cells are not themselves in the kernel. A moscot-only run on the full 9-stage
joint object is in `cellrank_joint_moscot.ipynb`.

**Kernels**
- `VelocityKernel` - directional, from RNA velocity (velocyto spliced/unspliced -> scVelo)
- `RealTimeKernel` - directional, from moscot optimal transport between consecutive stages
- `ConnectivityKernel` - undirected, a symmetric smoother; never used alone for direction

**Runs.** Each kernel alone, then five weightings, all with identical estimator settings so
the comparison is like-for-like. The question each one answers: does it find an e7.0 root,
and do the terminal states agree?

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import cellrank as cr
import matplotlib.pyplot as plt

print("cellrank", cr.__version__, "| scvelo", scv.__version__, "| scanpy", sc.__version__)
sc.settings.verbosity = 2
scv.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, dpi_save=200, frameon=False, figsize=(5, 5))

# --- compatibility shim -------------------------------------------------
# scanpy 1.11.5 calls .getnnz() on the kNN distance matrix, but scipy >= 1.16
# hands back a csr_*array* (sparse arrays dropped getnnz), so sc.pp.neighbors
# raises AttributeError. Restore the method rather than pinning scipy, which
# would risk the numpy-2 ABI that petsc4py in this env is built against.
import scipy.sparse as _sp

if not hasattr(_sp.csr_array, "getnnz"):
    def _getnnz(self, axis=None):
        if axis is None:
            return int(self.nnz)
        if axis == 1:
            return np.diff(self.indptr)
        return np.bincount(self.indices, minlength=self.shape[1])
    for _cls in (_sp.csr_array, _sp.csc_array):
        _cls.getnnz = _getnnz
    print("patched sparse-array getnnz for scanpy/scipy compatibility")
# ------------------------------------------------------------------------


def as_sparse_matrices(adata):
    """Force obsp/layers to scipy sparse *matrices*.

    anndata >= 0.12 hands back csr_*arrays*, but scvelo 0.3.4 still uses
    matrix-only APIs (.A1, .getnnz), so it crashes on them. Convert at the
    source instead of patching each call site. Re-run after anything that
    rebuilds these (subsetting, copying).
    """
    from scipy.sparse import csr_matrix, issparse, spmatrix
    for store in (adata.obsp, adata.layers):
        for k in list(store.keys()):
            v = store[k]
            if issparse(v) and not isinstance(v, spmatrix):
                store[k] = csr_matrix(v)
    return adata

In [ ]:
base = "/project/imoskowitz/yubin/Trajectory_inference/"
fig_cr = base + "output_fig/07_cellrank/"
res_dir = base + "data/cellrank/"
for d in (fig_cr, res_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = fig_cr
scv.settings.figdir = fig_cr

BASIS = "X_umap_joint"          # joint Harmony UMAP, for plotting
REP = "X_pca_harmony_joint"     # joint Harmony PCA, for graphs
N_STATES = 12                   # macrostates, fixed across runs for comparability

In [ ]:
adata = sc.read_h5ad(base + "data/devbrain_velocity.h5ad")
print(adata)
print("\nlayers:", list(adata.layers.keys()))
print("spliced total  :", f"{int(adata.layers['spliced'].sum()):,}")
print("unspliced total:", f"{int(adata.layers['unspliced'].sum()):,}")
print("\ncells per stage:")
print(adata.obs["age"].value_counts().sort_index().to_string())

## Is the devbrain-only graph connected on the joint embedding?

The joint run connected e7.0 through *Brooks* cells. Removing them may re-isolate it. This is
checked up front, because it bounds what the velocity kernel can say about the early lineage.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, use_rep=REP)
as_sparse_matrices(adata)

from scipy.sparse.csgraph import connected_components
n_comp, labels = connected_components(adata.obsp["connectivities"], directed=False)
sizes = pd.Series(labels).value_counts()
print(f"{n_comp} connected components; largest = {100*sizes.iloc[0]/adata.n_obs:.1f}% of cells")
adata.obs["component"] = labels

A = (adata.obsp["connectivities"] > 0).astype(np.float32)
cats = list(adata.obs["age"].cat.categories)
onehot = np.zeros((adata.n_obs, len(cats)), dtype=np.float32)
onehot[np.arange(adata.n_obs), adata.obs["age"].cat.codes.values] = 1
frac = A @ onehot
frac = frac / np.maximum(frac.sum(1, keepdims=True), 1)
mix = pd.DataFrame([frac[(adata.obs["age"] == c).values].mean(0) for c in cats],
                   index=cats, columns=cats)
print("\nage mixing on the devbrain-only graph built from the joint embedding:")
print((100 * mix).round(1).to_string())

## scVelo: moments and velocity

In [ ]:
HH_GENES = ["Gli1", "Hhip", "Smo", "Shh", "Ptch1"]

# scvelo 0.3.4 dropped n_top_genes/log/subset_highly_variable from
# filter_and_normalize - it now only filters and normalizes, so the log and HVG
# steps come from scanpy. X is reset to raw counts first: it arrives
# log-normalized from the processed object, and scvelo expects counts.
adata.X = adata.layers["counts"].copy()
scv.pp.filter_and_normalize(adata, min_shared_counts=20, retain_genes=HH_GENES)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

# retain_genes keeps the Hh genes through the count filter; this keeps them
# through the HVG subset too (Gli1 and Shh are too sparse to be HVGs)
keep = np.asarray(adata.var["highly_variable"]) | np.asarray(adata.var_names.isin(HH_GENES))
adata = adata[:, keep].copy()
print(f"after prep: {adata.shape}")
print("Hh genes retained:", [g for g in HH_GENES if g in adata.var_names])

as_sparse_matrices(adata)          # the gene subset above rebuilt layers/obsp
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
# scvelo 0.3.4's stochastic mode hits a numpy>=2 error in leastsq_generalized
# (assigning a size-1 array into a scalar slot). Patched locally in this env:
#   envs/cellrank_traj/.../scvelo/tools/optimization.py  (.orig backup alongside)
# The dynamical model would take hours on 68k cells and is only worth it if
# these velocities look coherent first.
as_sparse_matrices(adata)
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata, n_jobs=int(os.environ.get("SLURM_CPUS_PER_TASK", 8)))
print("velocity genes:", int(adata.var["velocity_genes"].sum()))

In [ ]:
scv.pl.velocity_embedding_stream(
    adata, basis=BASIS.replace("X_", ""), color="age", legend_loc="right margin",
    title="RNA velocity stream, joint Harmony UMAP", save="velocity_stream_age.png",
    show=False,
)
scv.tl.velocity_confidence(adata)
print(adata.obs[["velocity_length", "velocity_confidence"]].describe().round(2).to_string())

## moscot: optimal transport between consecutive stages

In [ ]:
from moscot.problems.time import TemporalProblem

tp = TemporalProblem(adata)
tp = tp.prepare(time_key="stage_num", joint_attr={"attr": "obsm", "key": REP})
print(tp)

In [ ]:
tp = tp.solve(epsilon=1e-2, tau_a=0.95, tau_b=0.95, scale_cost="mean")
for k, v in tp.solutions.items():
    print(f"  {k}: converged={v.converged}  cost={float(v.cost):.4g}")

## Build the three kernels

In [ ]:
vk = cr.kernels.VelocityKernel(adata).compute_transition_matrix()
print(vk)

ck = cr.kernels.ConnectivityKernel(adata).compute_transition_matrix()
print(ck)

rtk = cr.kernels.RealTimeKernel.from_moscot(tp)
rtk = rtk.compute_transition_matrix(self_transitions="all", conn_weight=0.2)
print(rtk)

## One estimator run per kernel / weighting

Identical settings everywhere: the same `n_states`, the same cluster key, the same
`X_umap_joint` for plotting. Terminal *and* initial states are predicted - the initial state
is the point of this exercise, so it is reported explicitly along with which stage it falls in.

In [ ]:
KERNELS = {
    "velocity":            vk,
    "realtime_moscot":     rtk,
    "connectivity":        ck,
    "vk0.8_ck0.2":         0.8 * vk + 0.2 * ck,
    "vk0.8_rtk0.2":        0.8 * vk + 0.2 * rtk,
    "vk0.5_rtk0.5":        0.5 * vk + 0.5 * rtk,
    "vk0.2_rtk0.8":        0.2 * vk + 0.8 * rtk,
    "vk0.6_rtk0.2_ck0.2":  0.6 * vk + 0.2 * rtk + 0.2 * ck,
}
print("\n".join(f"{k:22s} {type(v).__name__}" for k, v in KERNELS.items()))

In [ ]:
def run_estimator(name, kernel, n_states=N_STATES, cluster_key="leiden_joint"):
    """GPCCA on one kernel: macrostates, terminal + initial states, fate probabilities."""
    print("=" * 70, f"\n{name}\n", "=" * 70, flush=True)
    g = cr.estimators.GPCCA(kernel)
    g.compute_schur(n_components=n_states + 8, method="krylov")
    g.compute_macrostates(n_states=n_states, cluster_key=cluster_key)
    g.predict_terminal_states()
    try:
        g.predict_initial_states(n_states=1)
        init = list(g.initial_states.cat.categories)
    except Exception as e:                     # some kernels give no clean initial state
        print(f"  initial-state prediction failed: {e}")
        init = []
    term = list(g.terminal_states.cat.categories)
    print(f"  terminal states: {term}")
    print(f"  initial states : {init}")

    # which stage does each macrostate sit in?
    ms = g.macrostates
    comp = (pd.crosstab(ms[~ms.isna()], adata.obs.loc[~ms.isna(), "age"], normalize="index")
            * 100).round(1)
    print("\n  macrostate composition by stage (%):")
    print(comp.to_string())

    g.compute_fate_probabilities()
    fate = pd.DataFrame(np.asarray(g.fate_probabilities),
                        columns=list(g.fate_probabilities.names), index=adata.obs_names)

    g.plot_macrostates(which="all", basis=BASIS, legend_loc="right",
                       title=f"{name}: macrostates", save=f"{name}_macrostates.png")
    g.plot_macrostates(which="terminal", basis=BASIS, legend_loc="right",
                       title=f"{name}: terminal states", save=f"{name}_terminal.png")
    if init:
        g.plot_macrostates(which="initial", basis=BASIS, legend_loc="right",
                           title=f"{name}: initial state", save=f"{name}_initial.png")
    return {"name": name, "terminal": term, "initial": init, "composition": comp,
            "fate": fate, "macrostates": ms}


results = {}
for name, kern in KERNELS.items():
    try:
        results[name] = run_estimator(name, kern)
    except Exception as e:
        print(f"!! {name} FAILED: {type(e).__name__}: {e}", flush=True)
        results[name] = {"name": name, "error": f"{type(e).__name__}: {e}"}

## Compare the runs

In [ ]:
rows = []
for name, r in results.items():
    if "error" in r:
        rows.append({"kernel": name, "status": "FAILED", "n_terminal": None,
                     "initial_state": r["error"][:40], "initial_stage": None})
        continue
    init_stage = None
    if r["initial"]:
        ms = r["macrostates"]
        sel = ms.isin(r["initial"])
        if sel.any():
            init_stage = (adata.obs.loc[sel.values, "age"].value_counts(normalize=True)
                          .idxmax())
    rows.append({"kernel": name, "status": "ok", "n_terminal": len(r["terminal"]),
                 "initial_state": ", ".join(r["initial"]) if r["initial"] else "none",
                 "initial_stage": init_stage})
overview = pd.DataFrame(rows)
print(overview.to_string(index=False))
overview.to_csv(res_dir + "kernel_overview.csv", index=False)

In [ ]:
# do the weightings agree on fate? correlate fate probabilities between runs
ok = {n: r for n, r in results.items() if "error" not in r}
names = list(ok)
agree = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        fa, fb = ok[a]["fate"], ok[b]["fate"]
        shared = [c for c in fa.columns if c in fb.columns]
        if not shared:
            agree.loc[a, b] = np.nan
            continue
        agree.loc[a, b] = float(np.mean([
            np.corrcoef(fa[c].values, fb[c].values)[0, 1] for c in shared
        ]))
print("mean correlation of fate probabilities over shared terminal states:")
print(agree.astype(float).round(3).to_string())
agree.to_csv(res_dir + "fate_probability_agreement.csv")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(agree.astype(float).values, cmap="RdYlBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
for i in range(len(names)):
    for j in range(len(names)):
        v = agree.astype(float).values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Fate-probability agreement between kernel weightings", fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout()
fig.savefig(fig_cr + "kernel_agreement.png", dpi=200, bbox_inches="tight")

In [ ]:
# terminal-state overlap: do different weightings nominate the same clusters?
term_sets = {n: set(r["terminal"]) for n, r in ok.items()}
print("terminal states per run:")
for n, s in term_sets.items():
    print(f"  {n:22s} {sorted(s)}")
allt = sorted(set().union(*term_sets.values())) if term_sets else []
tab = pd.DataFrame({n: [t in s for t in allt] for n, s in term_sets.items()}, index=allt)
print("\nterminal state x run (True = nominated):")
print(tab.to_string())
tab.to_csv(res_dir + "terminal_state_overlap.csv")

## Save

In [ ]:
import resource
print(f"peak RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

for name, r in results.items():
    if "error" in r:
        continue
    r["fate"].to_csv(res_dir + f"fate_{name}.csv")
    adata.obs[f"macro_{name}"] = r["macrostates"]

adata.write_h5ad(base + "data/devbrain_cellrank.h5ad", compression="gzip")
print("wrote data/devbrain_cellrank.h5ad")